#Finalised Architecture

```mermaid
flowchart TD

    A[Landsat 8/9 Dataset] --> B[Co-registration & Alignment]

    B --> C[Thermal IR Bands]

    C --> D[Restormer Denoising Module]

    D --> E[SwinIR Super-Resolution Module]

    E --> F[Enhanced IR Image]

    F --> G[SegFormer Semantic Segmentation]
    F --> H[DINOv2 Semantic Feature Extraction]

    G --> I[Land Cover Masks]
    H --> J[Semantic Feature Embeddings]

    I --> K[Semantic Guided Pix2PixHD]
    J --> K

    K --> L[RGB Reconstruction]

    L --> M[Refinement Head]

    M --> N[Final High-Resolution RGB Output]

    N --> O[YOLOv11 / RT-DETR Validation]

    O --> P[Detection Consistency Loss]
```

##LuminaIR

## End-to-End Pipeline

```mermaid
flowchart TD

    A[Landsat 8/9 Thermal Bands] --> B[Preprocessing & Co-registration]

    B --> C[Restormer Denoising]

    C --> D[SwinIR Super-Resolution]

    D --> E[Enhanced Infrared Image]

    E --> F[SegFormer]
    E --> G[DINOv2]

    F --> H[Semantic Land-Cover Masks]
    G --> I[Semantic Feature Embeddings]

    H --> J[Semantic Guided Pix2PixHD Generator]
    I --> J

    J --> K[RGB Reconstruction]

    K --> L[Refinement Head]

    L --> M[Final Colorized RGB Image]

    M --> N[YOLOv11 / RT-DETR]

    N --> O[Detection Consistency Loss]

    O --> P[Model Optimization]
```

## Components

### 1. Data Preparation
- Landsat 8/9 Thermal Bands (B10, B11)
- RGB Bands (B4, B3, B2)
- Co-registration and alignment

### 2. Enhancement Module
- Restormer for denoising
- SwinIR for super-resolution

### 3. Semantic Understanding
- SegFormer for land-cover segmentation
- DINOv2 for semantic feature extraction

### 4. Colorization Module
- Semantic Guided Pix2PixHD
- RGB image reconstruction

### 5. Refinement Module
- Artifact removal
- Edge sharpening
- Texture enhancement

### 6. Downstream Validation
- YOLOv11 / RT-DETR
- Detection Consistency Loss

## Evaluation Metrics
- PSNR
- SSIM
- FID
- mAP Improvement
- Inference Time
```

1. The Restormer + SwinIR Combo (Perfect for Thermal Data)
This is a brilliant addition. Thermal IR bands captured at a 100-meter resolution are not just blurry; they are often incredibly noisy. If you just use SwinIR on raw thermal data, it might "super-resolve" the noise, creating sharp artifacts. Placing a Restormer (which is State-of-the-Art for image restoration and denoising) before SwinIR ensures you are only upscaling clean, structural thermal signatures.

2. Solving the Inference Time Trap
Standard Diffusion models require 20 to 50 denoising steps per image, taking seconds or minutes per tile. Pix2PixHD is a single-pass Generative Adversarial Network (GAN). It translates the image in one forward pass. This guarantees your pipeline will process images blazingly fast, satisfying the operational efficiency requirement for real-world deployment.

3. Retaining the Semantic Guardrails
You kept the SegFormer + DINOv2 branch. This is crucial because generative networks like GANs are highly prone to hallucinating (e.g., coloring a thermal roof green like a forest). Using these models to feed semantic masks into Pix2PixHD guarantees the "Semantic Integrity" objective perfectly.

Important Points to Note :
1. The VRAM Squeeze is Still Real: Even without Diffusion, running Restormer, SwinIR, SegFormer, DINOv2, a heavy GAN like Pix2PixHD, and YOLOv11 is a massive load. A free 16GB Colab GPU will still crash instantly.  have to freeze the weights of SegFormer, DINOv2, and YOLO during the main training loop, updating only the Pix2PixHD and Refinement Head weights.
2. The "Loss Balancing" Nightmare: Pix2PixHD already uses a complex, multi-scale GAN loss and feature-matching loss to train. If you add a Detection Consistency Loss from YOLOv11 on top of that, your loss function is going to be a chaotic math equation. If the YOLO loss is weighted too heavily, the GAN might generate weird, blocky artifacts that YOLO likes, but that look terrible to the human eye. You will need to carefully tune the $\lambda$ (lambda) weights for each loss component.